In [1]:
import numpy as np
from collections import Counter
import pandas as pd
from sklearn import decomposition
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math 
import heapq # priority data structure 

In [2]:
# Implement a two-dimensional k-nearest neighbors classifier 
# asked ChatGPT how to implement code in slides to fit problem needs
# from the slides - this is naive and testing against every point 


def knn_no_quad(test_pt, training_pts, training_types, k): # takes in point want to classify, list of known data points, class labels, how many nearest neighbors
    test_pt = np.array(test_pt) # takes in test points and turns into numpy array
    distances = [np.linalg.norm(test_pt - pt) for pt in training_pts] # calculates Eculidean distance from test point to each training point
    k_smallest_indices = np.argpartition(distances, k)[:k] # finds smallest distance/which training points are clostest to target
    k_nearest_labels = [training_types[i] for i in k_smallest_indices] # returns class label of nearest points
    most_common_label = Counter(k_nearest_labels).most_common(1)[0][0] # gives vote to closest neighbors
    return most_common_label

In [3]:
# Quad tree, need to subdivide at x and y median

In [ ]:
# from GeekforGeeks: https://www.geeksforgeeks.org/dsa/quad-tree/
# USed ChatGPT to understand each class etc
# Used to hold details of a point

class Point: # coordinates of a point, given in x and y 
    def __init__(self, x, y, label):
        self.x = x
        self.y = y
        self.label = label # add to have class

# The objects that we want stored in the quadtree
class Node: # a node is point put some data - coordinates + position + data = "There is a data point called Tree A located at coordinates 4, 5"
    def __init__(self, pos, data):
        self.pos = pos
        self.data = data

# The main quadtree class
class Quad:
    def __init__(self, topL, botR): # top of the rectangle points
        self.topLeft = topL
        self.botRight = botR
        self.n = None
        self.topLeftTree = None
        self.topRightTree = None
        self.botLeftTree = None
        self.botRightTree = None
    def k_nearest(self, query_point, k, heap=None):
        if heap is None:
            heap = []  # max-heap using negative distance
        if self.n is not None:
            for node in self.n: # iterate over all nodes in the leaf
                # Compute distance to the node in this quad
                d = distance(query_point, node.pos)
                if len(heap) < k:
                    heapq.heappush(heap, (-d, self.n))  # use negative distance for max-heap
                else:
                    if -d > heap[0][0]:  # closer than farthest in heap
                        heapq.heappushpop(heap, (-d, self.n))

        # Recursively check all sub-quads that exist
        for sub_quad in [self.topLeftTree, self.topRightTree, self.botLeftTree, self.botRightTree]:
            if sub_quad is not None:
                # Check if sub_quad could contain closer points
                closest_x = max(sub_quad.topLeft.x, min(query_point.x, sub_quad.botRight.x))
                closest_y = max(sub_quad.topLeft.y, min(query_point.y, sub_quad.botRight.y))
                closest_point_in_quad = Point(closest_x, closest_y, None)
                d_to_quad = distance(query_point, closest_point_in_quad)

                # Only recurse if this quad could have points closer than current farthest
                if len(heap) < k or d_to_quad < -heap[0][0]:
                    sub_quad.k_nearest(query_point, k, heap)

        if heap is None:
            return []
        else:
            # Return sorted list: closest first
            return sorted([(-d, node) for d, node in heap], key=lambda x: x[0])
    

    # Insert a node into the quadtree
    def insert(self, node):
        if node is None:
            return

        # Current quad cannot contain it
        if not self.inBoundary(node.pos):
            return

        # We are at a quad of unit area
        # We cannot subdivide this quad further
        if abs(self.topLeft.x - self.botRight.x) <= 1 and abs(self.topLeft.y - self.botRight.y) <= 1: # leaf nodes store points not parent
            if self.n is None:
                self.n = [node]
            else:  
                self.n.append(node)
            return

        if (self.topLeft.x + self.botRight.x) / 2 >= node.pos.x: # goes top left to bottom left 
            # Indicates topLeftTree
            if (self.topLeft.y + self.botRight.y) / 2 >= node.pos.y: # is doing recursive construction, creating child Quads that are recursively called later 
                if self.topLeftTree is None:
                    self.topLeftTree = Quad(self.topLeft, Point((self.topLeft.x + self.botRight.x) / 2, (self.topLeft.y + self.botRight.y) / 2, None))
                self.topLeftTree.insert(node)
            # Indicates botLeftTree
            else:
                if self.botLeftTree is None:
                    self.botLeftTree = Quad(Point(self.topLeft.x, (self.topLeft.y + self.botRight.y) / 2, None), Point((self.topLeft.x + self.botRight.x) / 2, self.botRight.y, None))
                self.botLeftTree.insert(node)
        else:
            # Indicates topRightTree # goes right top to bottom right 
            if (self.topLeft.y + self.botRight.y) / 2 >= node.pos.y:
                if self.topRightTree is None:
                    self.topRightTree = Quad(Point((self.topLeft.x + self.botRight.x) / 2, self.topLeft.y, None), Point(self.botRight.x, (self.topLeft.y + self.botRight.y) / 2, None))
                self.topRightTree.insert(node)
            # Indicates botRightTree
            else:
                if self.botRightTree is None:
                    self.botRightTree = Quad(Point((self.topLeft.x + self.botRight.x) / 2, (self.topLeft.y + self.botRight.y) / 2, None), self.botRight)
                self.botRightTree.insert(node)

    # Find a node in a quadtree
    def search(self, p):
        # Current quad cannot contain it
        if not self.inBoundary(p):
            return 0  # Return 0 if point is not found

        # We are at a quad of unit length
        # We cannot subdivide this quad further
        if self.n is not None:
            return self.n

        if (self.topLeft.x + self.botRight.x) / 2 >= p.x: # is doing recursive construction 
            # Indicates topLeftTree
            if (self.topLeft.y + self.botRight.y) / 2 >= p.y:
                if self.topLeftTree is None:
                    return 0
                return self.topLeftTree.search(p)
            # Indicates botLeftTree
            else:
                if self.botLeftTree is None:
                    return 0
                return self.botLeftTree.search(p)
        else:
            # Indicates topRightTree
            if (self.topLeft.y + self.botRight.y) / 2 >= p.y:
                if self.topRightTree is None:
                    return 0
                return self.topRightTree.search(p)
            # Indicates botRightTree
            else:
                if self.botRightTree is None:
                    return 0
                return self.botRightTree.search(p)

    # Check if current quadtree contains the point
    def inBoundary(self, p):
        return p.x >= self.topLeft.x and p.x <= self.botRight.x and p.y >= self.topLeft.y and p.y <= self.botRight.y



In [6]:
# test quad tree

def distance(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

# Example data: (x, y, label)
data_points = [
    (2, 3, "A"),
    (5, 4, "B"),
    (9, 6, "A"),
    (4, 7, "B"),
    (8, 1, "A"),
    (7, 2, "B")
]

# Define the quadtree boundaries (covers all x,y)
quad = Quad(Point(0, 0, 'A'), Point(10, 10, 'A'))

# Insert all data points as Nodes
for x, y, label in data_points:
    p = Point(x, y, label)
    node = Node(p, label)
    quad.insert(node)

query = Point(6, 3, 'A')
k = 3

neighbors = quad.k_nearest(query, k)
print("Nearest neighbors to", (query.x, query.y))
for dist, node in neighbors:
    print(f"  Label={node.data}, Pos=({node.pos.x}, {node.pos.y}), Dist={dist:.2f}")



TypeError: '<' not supported between instances of 'Node' and 'Node'

In [ ]:
# Implement a two-dimensional k-nearest neighbors classifier and store data in quad tree 
# asked ChatGPT how to implement code in slides to fit problem needs

def classify_knn_quadtree(test_pt, quadtree, k, initial_radius=1.0, max_radius=50.0):  # takes in point want to classify, list of known data points, class labels, how many nearest neighbors
    test_point = Point(test_pt[0], test_pt[1])
    radius = initial_radius
    found_points = [] # stores found points

    # Gradually expand the search window until we have at least k points
    while len(found_points) < k and radius <= max_radius:
        search_area = Rectangle(test_point.x, test_point.y, radius, radius)
        found_points = quadtree.query(search_area)
        radius *= 2  # exponentially expand region if too few points found
    if len(found_points) == 0:
        raise ValueError("No nearby points found even after expanding search area.")
    
    # Compute distances to found points
    distances = [np.linalg.norm(np.array([test_point.x, test_point.y]) - np.array([p.x, p.y]))
        for p in found_points]
    
    # Find k nearest points
    k_smallest_indices = np.argpartition(distances, k)[:k]
    k_nearest_labels = [found_points[i].label for i in k_smallest_indices]
    
    # Determine most common label among nearest neighbors
    most_common_label = Counter(k_nearest_labels).most_common(1)[0][0]
    return most_common_label

In [ ]:
# call in rice
rice = pd.read_excel("Rice_Cammeo_Osmancik.xlsx")

In [ ]:
rice.head()

,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,15231,525.578979,229.749878,85.093788,0.928882,15617,0.572896,Cammeo
1,14656,494.311005,206.020065,91.730972,0.895405,15072,0.615436,Cammeo
2,14634,501.122009,214.106781,87.768288,0.912118,14954,0.693259,Cammeo
3,13176,458.342987,193.337387,87.448395,0.891861,13368,0.640669,Cammeo
4,14688,507.166992,211.743378,89.312454,0.906691,15262,0.646024,Cammeo


In [ ]:
# standardize 7 quantitative rice data
def standardize(series):
    return (series - series.mean()) / series.std()

rice_standard = rice[[c for c in rice.columns if c != "Class"]].apply(standardize)

In [ ]:
# reduce data to 2D using PCA - only done for 7 quantitative columns

pca = decomposition.PCA(n_components=2)
data_reduced = pca.fit_transform(rice_standard[[c for c in rice.columns if c != "Class"]])
pc0 = data_reduced[:, 0]
pc1 = data_reduced[:, 1]

In [ ]:
# plot PCA0 vs PCA1
# Used ChatGPT to fix errors thrown of using categorical classes in plotly

fig = px.scatter(
    x=pc0,
    y=pc1,
    color=rice['Class'],
    title="Rice Data (PC0 vs PC1)",
    labels={'x': 'PC0', 'y': 'PC1'},
    template='plotly_white'
)

fig.write_image('Original_PCA.png')
fig.show()
